<a href="https://colab.research.google.com/github/Arda-Avci/AI-Publisher/blob/main/Google_Colab_AI_Publisher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Sistem bağımlılıklarını ve TTS'i Python 3.12 uyumlu olacak şekilde kaynağından kuruyoruz
!apt-get install -y espeak-ng espeak
!pip install diffusers transformers hf_transfer accelerate imageio-ffmpeg Flask flask-ngrok pyngrok opencv-python-headless scipy
!pip install -q "torch<2.6" "torchaudio<2.6" "torchvision<0.21" --no-cache-dir
!pip install coqui-tts


In [ ]:
# Colab hücresine yapıştırın
from google.colab import files
uploaded = files.upload()  # karakter.wav seçin
import shutil
shutil.copy(list(uploaded.keys())[0], "/content/karakter.wav")
print("✅ Referans ses yüklendi!")


In [ ]:
"""
AI-Publisher Colab Sunucu - v3 (ModelScope T2V)
================================================
CogVideoX-2b, Colab ücretsiz T4 GPU'sunun 12.67GB RAM sınırını
inference sırasında aşıyor. Bu nedenle:

- VIDEO : damo-vilab/text-to-video-ms-1.7b (ModelScope)
          → Inference RAM: ~4-5 GB (T4'te kararlı)
- TTS   : coqui/XTTS-v2 (değişmedi)
- SFX   : cvssp/audioldm2 (değişmedi)
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import torch
import cv2
import numpy as np
from flask import Flask, request, jsonify, send_file
from diffusers import DiffusionPipeline
import scipy.io.wavfile as wavfile
import gc
import traceback
from pyngrok import ngrok

app = Flask(__name__)

# ── Global hata yakalayıcı ───────────────────────────────────────────────────
@app.errorhandler(Exception)
def handle_exception(e):
    print("❌ SUNUCU HATA DETAYI:")
    traceback.print_exc()
    return jsonify({"status": "error", "message": str(e)}), 500

print("🚀 Flask sunucusu Lazy Loading (ModelScope T2V) ile hazırlandı.")

# ── YARDIMCI: GPU belleğini temizle ─────────────────────────────────────────
def flush_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ── 1. VİDEO ÜRETİMİ (ModelScope - T4 Uyumlu) ───────────────────────────────
def generate_video_lazy(prompt: str) -> list:
    """
    damo-vilab/text-to-video-ms-1.7b ile metin → video üretir.
    Inference RAM: ~4-5 GB  (Colab T4 limitinde güvenli)
    Çıktı: frame listesi (numpy array)
    """
    flush_memory()

    print("🎬 Video motoru (ModelScope-1.7b) belleğe yükleniyor...")
    pipe = DiffusionPipeline.from_pretrained(
        "damo-vilab/text-to-video-ms-1.7b",
        torch_dtype=torch.float16,
        variant="fp16",
        low_cpu_mem_usage=True,
    )
    pipe.enable_model_cpu_offload()   # Katmanları sırayla GPU'ya taşır
    pipe.enable_vae_slicing()         # VAE decode belleğini böler

    print("🎬 Video üretimi başlatıldı...")
    try:
        with torch.inference_mode():
            output = pipe(
                prompt=prompt,
                num_frames=16,           # ~2 sn @8fps — T4 için optimize
                num_inference_steps=25,  # Kalite/hız dengesi
                height=256,
                width=256,
            )
        frames = output.frames[0]        # List[PIL.Image] veya ndarray
    except torch.cuda.OutOfMemoryError as exc:
        print(f"❌ GPU OOM: {exc}")
        del pipe
        flush_memory()
        raise RuntimeError(
            "GPU belleği yetersiz. Colab oturumunu yeniden başlatın ve tekrar deneyin."
        ) from exc
    except Exception as exc:
        print(f"❌ Video üretim hatası: {exc}")
        del pipe
        flush_memory()
        raise

    del pipe
    flush_memory()
    return frames   # List[PIL.Image]

# ── 2. TTS ───────────────────────────────────────────────────────────────────
_tts_model = None

def get_tts():
    global _tts_model
    if _tts_model is None:
        from TTS.api import TTS
        print("🎙️ XTTS modeli belleğe yükleniyor...")
        _tts_model = TTS(
            model_name="tts_models/multilingual/multi-dataset/xtts_v2",
            gpu=True
        )
    return _tts_model

# ── 3. SFX ───────────────────────────────────────────────────────────────────
def generate_sfx_lazy(prompt: str):
    from diffusers import AudioLDM2Pipeline
    flush_memory()

    print("🔊 Ses efekti motoru belleğe yükleniyor...")
    sfx_pipe = AudioLDM2Pipeline.from_pretrained(
        "cvssp/audioldm2",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    sfx_pipe.enable_model_cpu_offload()

    print("🔊 Ses efekti üretiliyor...")
    with torch.inference_mode():
        audio = sfx_pipe(
            prompt,
            audio_length_in_s=3.0,    # Video süresiyle hizalı
            num_inference_steps=20,
        ).audios

    del sfx_pipe
    flush_memory()
    return audio

# ── 4. LİP-SYNC (OpenCV tabanlı) ─────────────────────────────────────────────
def apply_lipsync(video_path: str, audio_path: str, output_path: str):
    """
    Ses genliğine göre karakterin alt yüz bölgesini hafifçe ölçekler.
    Gerçek bir lip-sync değil, görsel bir simülasyondur.
    """
    cap = cv2.VideoCapture(video_path)
    sample_rate, audio_data = wavfile.read(audio_path)
    audio_amplitude = np.abs(audio_data.astype(np.float32))
    if audio_amplitude.ndim > 1:
        audio_amplitude = audio_amplitude[:, 0]

    fps = cap.get(cv2.CAP_PROP_FPS) or 8.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        samples_per_frame = int(sample_rate / fps)
        start = frame_idx * samples_per_frame
        chunk = audio_amplitude[start : start + samples_per_frame]
        volume = float(np.mean(chunk)) if len(chunk) > 0 else 0.0

        if volume > 300:
            h, w = frame.shape[:2]
            y0, y1 = int(h * 0.65), int(h * 0.85)
            x0, x1 = int(w * 0.35), int(w * 0.65)
            mouth = frame[y0:y1, x0:x1].copy()
            scale = min(1.0 + volume / 30000.0, 1.18)
            expanded = cv2.resize(mouth, (0, 0), fx=1.0, fy=scale)
            rh, rw = min(expanded.shape[0], y1 - y0), min(expanded.shape[1], x1 - x0)
            frame[y0 : y0 + rh, x0 : x0 + rw] = expanded[:rh, :rw]

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

# ── YARDIMCI: PIL frame listesini geçici MP4'e dönüştür ─────────────────────
def frames_to_mp4(frames, path: str, fps: int = 8):
    """
    ModelScope'un döndürdüğü PIL.Image listesini MP4'e yazar.
    """
    import imageio
    imageio.mimwrite(path, [np.array(f) for f in frames], fps=fps, quality=8)

# ── API ROTASI ────────────────────────────────────────────────────────────────
LAST_VIDEO_PATH  = "/content/current_scene.mp4"
RAW_VIDEO_PATH   = "/content/raw_video.mp4"
AUDIO_PATH       = "/content/speech.wav"
SFX_PATH         = "/content/sfx.wav"

@app.route("/generate-media", methods=["POST"])
def generate_media():
    data              = request.get_json(force=True)
    video_prompt      = data.get("video_prompt", "")
    speech_text       = data.get("speech_text", "")
    sfx_prompt        = data.get("sfx_prompt", "")
    character_features= data.get("character_features", "")

    final_prompt = f"{character_features}, {video_prompt}" if character_features else video_prompt

    # 1. Video
    try:
        frames = generate_video_lazy(final_prompt)
    except RuntimeError as exc:
        return jsonify({"status": "error", "message": str(exc)}), 503

    frames_to_mp4(frames, RAW_VIDEO_PATH, fps=8)

    # 2. TTS
    if speech_text:
        speaker_wav = "/content/karakter.wav" if os.path.exists("/content/karakter.wav") else None
        tts = get_tts()
        tts.tts_to_file(
            text=speech_text,
            speaker_wav=speaker_wav,
            language="tr",
            file_path=AUDIO_PATH,
        )
    else:
        silence = np.zeros(int(16000 * 3), dtype=np.int16)  # 3 sn sessizlik
        wavfile.write(AUDIO_PATH, 16000, silence)

    # 3. Lip-Sync
    apply_lipsync(RAW_VIDEO_PATH, AUDIO_PATH, LAST_VIDEO_PATH)

    # 4. SFX
    if sfx_prompt:
        audio_sfx = generate_sfx_lazy(sfx_prompt)
        wavfile.write(SFX_PATH, 16000, (audio_sfx[0] * 32767).astype(np.int16))
    else:
        silence = np.zeros(int(16000 * 3), dtype=np.int16)
        wavfile.write(SFX_PATH, 16000, silence)

    return jsonify({"status": "success"})

# ── İNDİRME ROTALARI ─────────────────────────────────────────────────────────
@app.route("/download/video")
def download_video():
    return send_file(LAST_VIDEO_PATH, mimetype="video/mp4")

@app.route("/download/speech")
def download_speech():
    return send_file(AUDIO_PATH, mimetype="audio/wav")

@app.route("/download/sfx")
def download_sfx():
    return send_file(SFX_PATH, mimetype="audio/wav")

@app.route("/health")
def health():
    mem = {}
    if torch.cuda.is_available():
        mem["gpu_free_gb"]  = round(torch.cuda.mem_get_info()[0] / 1e9, 2)
        mem["gpu_total_gb"] = round(torch.cuda.mem_get_info()[1] / 1e9, 2)
    return jsonify({"status": "ok", "memory": mem})

# ── BAŞLATMA ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    NGROK_TOKEN = "3EYJuaphUgm2YMvhGpiyRR8OZa6_3cKw59pY6yKS4jxgohw76"
    if NGROK_TOKEN and NGROK_TOKEN != "BURAYA_NGROK_TOKEN_GELECEK":
        ngrok.set_auth_token(NGROK_TOKEN)
        public_url = ngrok.connect(5000)
        print("\n🔗 NODE.JS PROJENİZE YAPIŞTIRACAĞINIZ URL:\n", public_url.public_url)
        print("\n" + "-" * 50 + "\n")
    else:
        print("\n⚠️ NGROK_TOKEN girilmedi — yalnızca localhost.\n")

    app.run(port=5000, debug=True, use_reloader=False)
